# 07 · Ngưỡng stay-point: 200 m / 30 phút

**Câu hỏi:** bán kính 200 m và thời gian tối thiểu 30 phút (chế độ CENTROID) có hợp lý trên dữ liệu đã làm sạch, với cách chạy nối file (notebook 04), không?

**Nguồn gốc của 200 m / 30 phút:** đây là ngưỡng phổ biến trong các nghiên cứu trên GeoLife (thuật toán cửa sổ trượt của Li et al. 2008; Zheng et al. dùng 200 m với 20 hoặc 30 phút). Notebook cũ (`archive/04_find_threshold_stay_point_detector`) chỉ xem bản đồ trên dữ liệu thô bị nối file, nên chưa phải bằng chứng.

**Không có "đáp án đúng" cho stay-point**, nên dùng **nhãn thật** (`labels.txt`, 20% dữ liệu) theo hai hướng:
- **Đối chứng âm:** thời gian nằm trong một **chuyến đi có nhãn** là lúc người dùng đang di chuyển. Stay-point nằm gần trọn (≥ 80%) trong một chuyến đi là **stay-point giả**.
- **Đối chứng dương:** khoảng trống **≥ 60 phút** giữa hai chuyến đi có nhãn liên tiếp là lúc người dùng **ở đâu đó**. Bộ phát hiện nên phủ được phần lớn khoảng trống đó.

**Giới hạn của nhãn:** nhãn không phủ mọi chuyến đi. Một khoảng trống giữa hai chuyến đi có thể chứa một chuyến không được ghi nhãn, và một nhãn dài (ví dụ "walk" cả ngày) có thể chứa lúc dừng thật. Các tỉ lệ vì vậy là **ước lượng tương đối**, dùng để so sánh giữa các ngưỡng, không phải độ chính xác tuyệt đối.

**Dữ liệu:** chỉ dùng các chuỗi file (nối ≤ 18 h, notebook 04) của 69 user có nhãn mà có chứa ít nhất một chuyến đi có nhãn. Chỉ dùng điểm GPS thật.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK = "07_staypoint_thresholds"
DATA_DIR = Path("../data/raw")
OUT_DIR = Path("outputs") / NOTEBOOK
FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR = (OUT_DIR / d for d in ("figures", "tables", "cache", "maps"))
for d in (FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(Path("../src").resolve()))
from gps.data.processing import _haversine_vectorized
from gps.features.stay_point import MAX_GAP_HOURS, DistanceMode, StayPointDetector

USERS_DIR = Path("../data/processed/users")
MIN_GAP_MIN = 15        # stops between labelled trips kept from 15 min (bands in 3.1)
LONG_STOP_MIN = 60      # positive control of the main sweep: stops >= 60 min
VEHICLES = ["bus", "car", "taxi", "subway", "train"]
SHORT_TRIP_H = 3        # clean negative control: vehicle trips <= 3 h
MAX_GAP_H = MAX_GAP_HOURS
FALSE_SHARE = 0.8       # a stay >= 80% inside one labelled trip is a false stay


def load_labels(uid: str) -> pd.DataFrame | None:
    path = DATA_DIR / uid / "labels.txt"
    if not path.exists():
        return None
    lab = pd.read_csv(path, sep="\t", skiprows=1, header=None, names=["start", "end", "mode"])
    lab["start"] = pd.to_datetime(lab["start"], format="%Y/%m/%d %H:%M:%S", errors="coerce")
    lab["end"] = pd.to_datetime(lab["end"], format="%Y/%m/%d %H:%M:%S", errors="coerce")
    return lab.dropna().sort_values("start").reset_index(drop=True)

## 1. Dữ liệu: chuỗi file, chuyến đi có nhãn và khoảng dừng
Một chuyến đi hoặc khoảng dừng chỉ được dùng khi nằm trọn trong thời gian của một chuỗi file, tức bộ phát hiện có dữ liệu ở cả hai đầu.

In [ ]:
POINTS_PATH, TRIPS_PATH, GAPS_PATH = (CACHE_DIR / f for f in ("points.parquet", "trips.parquet", "gaps.parquet"))
if POINTS_PATH.exists():
    points, trips, gaps = pd.read_parquet(POINTS_PATH), pd.read_parquet(TRIPS_PATH), pd.read_parquet(GAPS_PATH)
else:
    p_parts, t_parts, g_parts = [], [], []
    for path in sorted(USERS_DIR.glob("user_*.parquet")):
        uid = path.stem.split("_")[1]
        lab = load_labels(uid)
        if lab is None:
            continue
        df = pd.read_parquet(path, columns=["source_file", "datetime", "lat", "lon"])
        bounds = df.groupby("source_file")["datetime"].agg(["min", "max"]).sort_values("min")
        bounds["chain"] = ((bounds["min"] - bounds["max"].shift()).dt.total_seconds() / 3600 > MAX_GAP_H).cumsum()
        chains = bounds.groupby("chain").agg(c_start=("min", "min"), c_end=("max", "max"))
        cs, ce = chains["c_start"].to_numpy(), chains["c_end"].to_numpy()

        def chain_of(t0, t1):
            k = np.searchsorted(cs, t0, side="right") - 1
            ok = (k >= 0) & (ce[np.maximum(k, 0)] >= t1)
            return np.where(ok, k, -1)

        lab["chain"] = chain_of(lab["start"].to_numpy(), lab["end"].to_numpy())
        nxt = lab["start"].shift(-1)
        gap = pd.DataFrame({"start": lab["end"], "end": nxt}).dropna()
        gap = gap[((gap["end"] - gap["start"]).dt.total_seconds() / 60 >= MIN_GAP_MIN)
                  & ((gap["end"] - gap["start"]).dt.total_seconds() / 3600 <= MAX_GAP_H)]
        gap["chain"] = chain_of(gap["start"].to_numpy(), gap["end"].to_numpy())
        used = set(lab.loc[lab["chain"] >= 0, "chain"])
        keep_files = bounds.index[bounds["chain"].isin(used)]
        p_parts.append(df[df["source_file"].isin(keep_files)].assign(user_id=uid))
        t_parts.append(lab[lab["chain"] >= 0].assign(user_id=uid))
        g_parts.append(gap[gap["chain"] >= 0].assign(user_id=uid))
    points, trips, gaps = (pd.concat(x, ignore_index=True) for x in (p_parts, t_parts, g_parts))
    points.to_parquet(POINTS_PATH)
    trips.to_parquet(TRIPS_PATH)
    gaps.to_parquet(GAPS_PATH)

trip_hours = ((trips["end"] - trips["start"]).dt.total_seconds() / 3600).sum()
gaps["minutes"] = (gaps["end"] - gaps["start"]).dt.total_seconds() / 60
long_gaps = gaps[gaps["minutes"] >= LONG_STOP_MIN]
vehicle_trips = trips[trips["mode"].isin(VEHICLES) & ((trips["end"] - trips["start"]).dt.total_seconds() / 3600 <= SHORT_TRIP_H)]
vehicle_hours = ((vehicle_trips["end"] - vehicle_trips["start"]).dt.total_seconds() / 3600).sum()
print(f"Users: {points['user_id'].nunique()} | points: {len(points):,} | labelled trips: {len(trips):,} "
      f"({trip_hours:,.0f} h; vehicle trips <= {SHORT_TRIP_H} h: {vehicle_hours:,.0f} h) | "
      f"stops >= {MIN_GAP_MIN} min between trips: {len(gaps):,} (>= {LONG_STOP_MIN} min: {len(long_gaps):,})")

## 2. Điểm GPS lúc đứng yên phân tán bao xa?
Lấy các khoảng dừng (≥ 60 phút giữa hai chuyến đi có nhãn) có ít nhất 10 điểm GPS bên trong. Với mỗi khoảng, đo khoảng cách từ các điểm tới **vị trí trung vị** của khoảng đó. `p50_m` là "điểm điển hình lệch bao xa", `p90_m` là "phần lớn điểm nằm trong bán kính nào". Bán kính stay-point phải lớn hơn độ phân tán này, nếu không một lần ở yên sẽ bị cắt thành nhiều mảnh.

In [ ]:
rows = []
for uid, g in long_gaps.groupby("user_id"):
    p = points[points["user_id"] == uid].sort_values("datetime")
    ts = p["datetime"].to_numpy()
    for s, e in zip(g["start"].to_numpy(), g["end"].to_numpy()):
        a, b = np.searchsorted(ts, s, "left"), np.searchsorted(ts, e, "right")
        if b - a < 10:
            continue
        la, lo = p["lat"].to_numpy()[a:b], p["lon"].to_numpy()[a:b]
        d = _haversine_vectorized(la, lo, np.full(b - a, np.median(la)), np.full(b - a, np.median(lo)))
        rows.append({"user_id": uid, "n_points": b - a, "p50_m": np.median(d), "p90_m": np.quantile(d, 0.9)})
footprint = pd.DataFrame(rows)
summary = footprint[["p50_m", "p90_m"]].quantile([0.5, 0.75, 0.9]).T
summary.columns = ["median_over_stops", "p75_over_stops", "p90_over_stops"]
summary["share_within_200m"] = [(footprint["p50_m"] <= 200).mean(), (footprint["p90_m"] <= 200).mean()]
summary.to_csv(TABLE_DIR / "stationary_footprint.csv")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.log10(footprint["p90_m"].clip(lower=1)), bins=60, color="#4f81bd")
for r in (50, 100, 200, 300, 500):
    ax.axvline(np.log10(r), color="red" if r == 200 else "gray", ls="--", lw=1)
    ax.text(np.log10(r), ax.get_ylim()[1] * 0.95, f"{r} m", rotation=90, va="top", ha="right", fontsize=8)
ax.set_xlabel("log10(P90 distance of points to the stop's median position, m)")
ax.set_ylabel("Stops between labelled trips")
ax.set_title(f"Footprint of stationary periods (n = {len(footprint):,} stops >= {LONG_STOP_MIN} min)")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig01_stationary_footprint.png", dpi=150)
plt.show()
summary.round(2)

## 3. Quét bán kính và thời gian
Mỗi tổ hợp chạy **chính bộ phát hiện của `src/`** (`detect()`, CENTROID, cắt tại khoảng ngắt > 18 h) trên toàn bộ dữ liệu ở mục 1:
- **Quét bán kính** 50–500 m với thời gian giữ 30 phút;
- **Quét thời gian** 5–60 phút với bán kính giữ 200 m.

Các chỉ số:
- `stop_recall`: tỉ lệ khoảng dừng (≥ 60 phút giữa hai chuyến đi) có **≥ 50%** thời gian được stay-point phủ;
- `false_stays_per_100_trip_h`: số stay-point nằm ≥ 80% trong một chuyến đi có nhãn, trên 100 giờ di chuyển có nhãn;
- `false_share`: tỉ lệ stay-point giả trong các stay-point chạm vào thời gian có nhãn (chuyến đi hoặc khoảng dừng).

Mỗi tổ hợp mất vài phút và được cache riêng.

In [ ]:
GRID = [(r, 30) for r in (50, 100, 150, 200, 300, 500)] + [(200, t) for t in (5, 10, 15, 20, 45, 60)]


def detect_all(radius_m: int, minutes: int) -> pd.DataFrame:
    path = CACHE_DIR / f"stays_r{radius_m}_t{minutes}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    det = StayPointDetector(minutes, radius_m, DistanceMode.CENTROID)
    rows = []
    for uid, p in points.groupby("user_id"):
        for sp in det.detect(p):
            rows.append({"user_id": uid, "arrival": pd.Timestamp(sp.arrival_time), "departure": pd.Timestamp(sp.departure_time),
                         "lat": sp.lat, "lon": sp.lon, "duration_min": sp.duration_minutes})
    out = pd.DataFrame(rows)
    out.to_parquet(path)
    return out


def overlap_minutes(s0, s1, a0, a1):
    """Overlap (min) between each interval [s0, s1] and the union of sorted, disjoint intervals [a0, a1]."""
    out = np.zeros(len(s0))
    for i, (x, y) in enumerate(zip(s0, s1)):
        k0, k1 = np.searchsorted(a1, x, "left"), np.searchsorted(a0, y, "right")
        if k1 > k0:
            lo, hi = np.maximum(a0[k0:k1], x), np.minimum(a1[k0:k1], y)
            out[i] = np.clip((hi - lo) / np.timedelta64(1, "m"), 0, None).sum()
    return out


def evaluate(stays: pd.DataFrame) -> tuple[dict, np.ndarray]:
    covered, false_n, touch_n, false_vehicle = [], 0, 0, 0
    for uid, g in gaps.groupby("user_id"):
        s = stays[stays["user_id"] == uid].sort_values("arrival")
        cov = overlap_minutes(g["start"].to_numpy(), g["end"].to_numpy(), s["arrival"].to_numpy(), s["departure"].to_numpy())
        covered += list(cov / ((g["end"] - g["start"]).dt.total_seconds() / 60).to_numpy())
    for uid, s in stays.groupby("user_id"):
        t = trips[trips["user_id"] == uid].sort_values("start")
        g = gaps[gaps["user_id"] == uid].sort_values("start")
        dur = s["duration_min"].to_numpy()
        v = vehicle_trips[vehicle_trips["user_id"] == uid]
        in_vehicle = np.zeros(len(s))
        for t0, t1 in zip(v["start"].to_numpy(), v["end"].to_numpy()):
            in_vehicle = np.maximum(in_vehicle, np.clip((np.minimum(s["departure"].to_numpy(), t1)
                                                         - np.maximum(s["arrival"].to_numpy(), t0)) / np.timedelta64(1, "m"), 0, None))
        false_vehicle += int((in_vehicle >= FALSE_SHARE * np.maximum(dur, 1e-9)).sum())
        in_trip = np.zeros(len(s))
        for t0, t1 in zip(t["start"].to_numpy(), t["end"].to_numpy()):   # max overlap with ONE trip
            ov = np.clip((np.minimum(s["departure"].to_numpy(), t1) - np.maximum(s["arrival"].to_numpy(), t0))
                         / np.timedelta64(1, "m"), 0, None)
            in_trip = np.maximum(in_trip, ov)
        in_gap = overlap_minutes(s["arrival"].to_numpy(), s["departure"].to_numpy(), g["start"].to_numpy(), g["end"].to_numpy())
        false_n += int((in_trip >= FALSE_SHARE * np.maximum(dur, 1e-9)).sum())
        touch_n += int(((in_trip > 0) | (in_gap > 0)).sum())
    covered = np.array(covered)
    is_long = gaps.sort_values(["user_id"], kind="stable")["minutes"].to_numpy() >= LONG_STOP_MIN
    return {"stay_points": len(stays), "stop_recall": (covered[is_long] >= 0.5).mean(),
            "stop_time_covered": covered[is_long].mean(),
            "false_stays": false_n, "false_stays_per_100_trip_h": 100 * false_n / trip_hours,
            "false_share": false_n / max(touch_n, 1),
            "false_in_vehicle_per_100_h": 100 * false_vehicle / vehicle_hours}, covered


rows, band_rows = [], []
gap_order = gaps.sort_values(["user_id"], kind="stable")
BANDS = pd.cut(gap_order["minutes"], [15, 30, 45, 60, 120, MAX_GAP_H * 60], include_lowest=True,
               labels=["15-30", "30-45", "45-60", "60-120", "2-18h"])
for r, t in GRID:
    print(f"r = {r} m, t = {t} min", flush=True)
    metrics, covered = evaluate(detect_all(r, t))
    rows.append({"radius_m": r, "min_minutes": t, **metrics})
    band_rows.append({"radius_m": r, "min_minutes": t,
                      **pd.Series(covered >= 0.5).groupby(BANDS.to_numpy(), observed=False).mean().to_dict()})
sweep = pd.DataFrame(rows)
recall_by_stop_length = pd.DataFrame(band_rows)
recall_by_stop_length.to_csv(TABLE_DIR / "recall_by_stop_length.csv", index=False)
sweep.to_csv(TABLE_DIR / "threshold_sweep.csv", index=False)
sweep.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (key, fixed, label) in zip(axes, [("radius_m", ("min_minutes", 30), "Radius (m), time fixed at 30 min"),
                                          ("min_minutes", ("radius_m", 200), "Minimum duration (min), radius fixed at 200 m")]):
    d = sweep.merge(recall_by_stop_length, on=["radius_m", "min_minutes"])
    d = d[d[fixed[0]] == fixed[1]].sort_values(key)
    found, found_label = (("stop_recall", "Stops >= 60 min found (%)") if key == "radius_m"
                          else ("30-45", "Stops of 30-45 min found (%)"))
    ax.plot(d[key], 100 * d[found], "o-", color="#2ca02c", label=found_label)
    ax.set_ylim(0, 100)
    ax.set_ylabel("Stops found (%)", color="#2ca02c")
    ax2 = ax.twinx()
    ax2.plot(d[key], d["false_in_vehicle_per_100_h"], "s-", color="#d62728", label="False stays per 100 h in vehicles")
    ax2.set_yscale("log")
    ax2.set_ylabel("False stays per 100 h of vehicle trips (log)", color="#d62728")
    ax.axvline(200 if key == "radius_m" else 30, color="gray", ls="--", lw=1)
    ax.set_xlabel(label)
    ax.legend(loc="upper left", fontsize=8)
    ax2.legend(loc="lower right", fontsize=8)
axes[0].set_title("Radius sweep")
axes[1].set_title("Minimum duration sweep")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig02_threshold_sweep.png", dpi=150)
plt.show()

### 3.1 Tỉ lệ bắt được theo độ dài khoảng dừng
Mỗi ô là tỉ lệ khoảng dừng (giữa hai chuyến đi có nhãn) có ≥ 50% thời gian được stay-point phủ. Ngưỡng thời gian chỉ có thể ảnh hưởng tới các khoảng dừng gần bằng nó, nên phải nhìn theo từng nhóm độ dài.

In [ ]:
print("Stops per length band:", BANDS.value_counts().sort_index().to_dict())
recall_by_stop_length.round(3)

### 3.2 Stay-point giả nằm trong loại chuyến đi nào?
Với 200 m / 30 phút: stay-point giả được xếp theo phương tiện của chuyến đi chứa nó và theo độ dài chuyến đi.

In [ ]:
s = detect_all(200, 30)
rows = []
for uid, g in s.groupby("user_id"):
    tt = trips[trips["user_id"] == uid].reset_index(drop=True)
    a, d, dur = g["arrival"].to_numpy(), g["departure"].to_numpy(), g["duration_min"].to_numpy()
    best, idx = np.zeros(len(g)), np.full(len(g), -1)
    for k, (t0, t1) in enumerate(zip(tt["start"].to_numpy(), tt["end"].to_numpy())):
        ov = np.clip((np.minimum(d, t1) - np.maximum(a, t0)) / np.timedelta64(1, "m"), 0, None)
        upd = ov > best
        best[upd], idx[upd] = ov[upd], k
    for k in idx[best >= FALSE_SHARE * dur]:
        rows.append({"mode": tt.at[k, "mode"], "trip_h": (tt.at[k, "end"] - tt.at[k, "start"]).total_seconds() / 3600})
false_by_mode = pd.DataFrame(rows)
false_by_mode["trip_length"] = pd.cut(false_by_mode["trip_h"], [0, 1, 3, 6, np.inf], labels=["<1h", "1-3h", "3-6h", ">6h"])
false_table = pd.crosstab(false_by_mode["mode"], false_by_mode["trip_length"], margins=True)
false_table.to_csv(TABLE_DIR / "false_stays_by_trip_mode.csv")
false_table

## 4. Có nên cắt "đuôi" của cửa sổ CENTROID?
Ở chế độ CENTROID, mỗi điểm chỉ được so với **tâm tại lúc nó được thêm vào**. Tâm trôi dần về phía chỗ ở thật, nên các điểm đầu cửa sổ (lúc đang tới) và cuối cửa sổ (lúc bắt đầu rời đi) có thể nằm **xa hơn 200 m so với tâm cuối cùng**. Câu hỏi: có nên cắt các điểm đầu/cuối đó ra khỏi stay-point không?

Lấy chính các stay-point của `src/` (200 m / 30 phút) và so sánh ba cách báo cáo cùng một cửa sổ (vị trí quét tiếp theo không đổi):
- **T0**: giữ nguyên, như `src/`;
- **T1**: cắt một lần các điểm đầu và cuối nằm xa hơn 200 m so với tâm, rồi tính lại tâm;
- **T2**: lặp lại bước cắt tới khi không còn gì để cắt.

Stay-point nào còn dưới 30 phút sau khi cắt thì bị bỏ. Chỉ số gồm các chỉ số ở mục 3, và thêm **sai lệch mốc thời gian so với nhãn**: với mỗi khoảng dừng ≥ 30 phút giữa hai chuyến đi có nhãn, lấy stay-point chồng lên nó nhiều nhất, rồi đo giờ đến trừ giờ kết thúc chuyến trước, và giờ đi trừ giờ bắt đầu chuyến sau. Tất cả đều trên điểm GPS thật.

In [ ]:
TAILS_PATH = CACHE_DIR / "tail_trim_r200_t30.parquet"


def trim(lat: np.ndarray, lon: np.ndarray, repeat: bool) -> tuple[int, int]:
    a, b = 0, len(lat) - 1
    while True:
        d = _haversine_vectorized(lat, lon, np.full(len(lat), lat[a:b + 1].mean()), np.full(len(lon), lon[a:b + 1].mean()))
        a0, b0 = a, b
        while a < b and d[a] > 200:
            a += 1
        while b > a and d[b] > 200:
            b -= 1
        if not repeat or (a, b) == (a0, b0):
            return a, b


if TAILS_PATH.exists():
    tails = pd.read_parquet(TAILS_PATH)
else:
    rows = []
    base = detect_all(200, 30)
    for uid, s in base.groupby("user_id"):
        p = points[points["user_id"] == uid].sort_values("datetime")
        ts, la, lo = p["datetime"].to_numpy(), p["lat"].to_numpy(), p["lon"].to_numpy()
        for k, r in enumerate(s.itertuples()):
            i0, i1 = np.searchsorted(ts, np.datetime64(r.arrival), "left"), np.searchsorted(ts, np.datetime64(r.departure), "right")
            wl, wo, wt = la[i0:i1], lo[i0:i1], ts[i0:i1]
            for variant, (a, b) in (("T0", (0, len(wl) - 1)), ("T1", trim(wl, wo, False)), ("T2", trim(wl, wo, True))):
                cl, co = wl[a:b + 1].mean(), wo[a:b + 1].mean()
                rows.append({"variant": variant, "user_id": uid, "stay": k, "arrival": pd.Timestamp(wt[a]),
                             "departure": pd.Timestamp(wt[b]), "lat": cl, "lon": co,
                             "duration_min": (wt[b] - wt[a]) / np.timedelta64(1, "m"), "trimmed": (a, b) != (0, len(wl) - 1),
                             "max_dist_m": _haversine_vectorized(wl[a:b + 1], wo[a:b + 1], np.full(b - a + 1, cl), np.full(b - a + 1, co)).max(),
                             "shift_m": _haversine_vectorized(np.array([cl]), np.array([co]), np.array([r.lat]), np.array([r.lon]))[0]})
    tails = pd.DataFrame(rows)
    tails.to_parquet(TAILS_PATH)


def boundary_errors(st: pd.DataFrame) -> pd.DataFrame:
    out = []
    for uid, gg in gaps[gaps["minutes"] >= 30].groupby("user_id"):
        s = st[st["user_id"] == uid]
        for g0, g1 in zip(gg["start"], gg["end"]):
            ov = (s["departure"].clip(upper=g1) - s["arrival"].clip(lower=g0)).dt.total_seconds()
            if len(s) and ov.max() > 0:
                m = s.loc[ov.idxmax()]
                out.append({"arr_min": (m["arrival"] - g0).total_seconds() / 60, "dep_min": (m["departure"] - g1).total_seconds() / 60})
    return pd.DataFrame(out)


rows = []
for variant, st in tails.groupby("variant"):
    st = st[st["duration_min"] >= 30]
    m, _ = evaluate(st)
    b, tr = boundary_errors(st), st[st["trimmed"]]
    rows.append({"variant": variant, "stay_points": len(st), "trimmed": len(tr), "stop_recall": m["stop_recall"],
                 "false_in_vehicle_per_100_h": m["false_in_vehicle_per_100_h"],
                 "share_with_point_over_200m": (st["max_dist_m"] > 200).mean(), "max_dist_p90_m": st["max_dist_m"].quantile(0.9),
                 "centroid_shift_p50_m": tr["shift_m"].median() if len(tr) else 0.0, "centroid_shift_p90_m": tr["shift_m"].quantile(0.9) if len(tr) else 0.0,
                 "arrival_err_p50_min": b["arr_min"].median(), "arrival_abs_err_p50_min": b["arr_min"].abs().median(),
                 "departure_err_p50_min": b["dep_min"].median(), "departure_abs_err_p50_min": b["dep_min"].abs().median()})
tail_table = pd.DataFrame(rows).set_index("variant")
tail_table.to_csv(TABLE_DIR / "tail_trimming.csv")
tail_table.T.round(3)

## Kết luận

**Về đối chứng âm:** 75% stay-point "giả" (778/1,044 ở 200 m / 30 phút) nằm trong nhãn **walk**, phần lớn là nhãn dài 1 giờ tới hơn 6 giờ. Các nhãn này thường bao trùm cả lúc dừng thật (ở trường, ở chợ), nên `false_share` (khoảng 29%) **đánh giá quá cao** số stay-point giả. Đối chứng âm sạch hơn là **chuyến đi bằng xe ≤ 3 h** (bus, car, taxi, subway, train; 2,131 giờ): trong đó gần như không thể có lần dừng ≥ 30 phút thật. Kết luận dưới đây dựa trên chỉ số `false_in_vehicle_per_100_h`.

**Thời gian tối thiểu 30 phút có bằng chứng rõ:**

| Thời gian tối thiểu | 5 | 10 | 15 | 20 | **30** | 45 | 60 |
|---|---|---|---|---|---|---|---|
| Stay-point giả / 100 giờ đi xe | 102 | 28.4 | 13.2 | 7.3 | **3.5** | 1.6 | 0.8 |
| Bắt được khoảng dừng 30–45 phút | 54% | 54% | 56% | 56% | **53%** | 33% | 13% |

- Từ 5 lên 30 phút, số stay-point giả giảm khoảng 30 lần.
- 30 phút là mốc **lớn nhất** mà vẫn giữ được các khoảng dừng 30–45 phút. Lên 45 phút thì mất một phần ba số đó, lên 60 phút thì mất gần hết.
- Hạ xuống dưới 30 phút chỉ thêm được các khoảng dừng 15–30 phút (20% → 43%), nhưng stay-point giả tăng gấp 2 tới 30 lần. Với bài toán Home/Office (tính theo thời gian lưu trú), các lần dừng dưới 30 phút đóng góp rất ít.

**Bán kính 200 m hợp lý, nhưng dữ liệu không chỉ ra một mức tối ưu:**

| Bán kính (m) | 50 | 100 | 150 | **200** | 300 | 500 |
|---|---|---|---|---|---|---|
| Bắt được khoảng dừng ≥ 60 phút | 18% | 32% | 42% | **49%** | 58% | 69% |
| Stay-point giả / 100 giờ đi xe | 1.5 | 2.1 | 2.9 | **3.5** | 4.4 | 5.2 |

- Độ phân tán lúc đứng yên: ở khoảng dừng trung vị, 90% số điểm nằm trong **136 m** quanh vị trí trung vị, nên bán kính dưới khoảng 150 m sẽ cắt một lần ở yên thành nhiều mảnh. Đó là lý do 50–100 m chỉ bắt được 18–32%.
- Từ 200 m trở lên, tỉ lệ bắt được và stay-point giả **cùng tăng gần tuyến tính**, không có điểm gãy. Phần tăng thêm khi nới bán kính một phần là do gộp cả việc đi lại không được ghi nhãn trong khoảng dừng.
- 200 m bao được độ phân tán điển hình lúc đứng yên, giữ stay-point giả ở mức thấp (khoảng 1 stay-point giả trên 29 giờ đi xe), và vẫn tách được các địa điểm cách nhau vài trăm mét. 300 m cũng bảo vệ được; dữ liệu không đủ để phân định giữa hai mức này.

**Tỉ lệ bắt được tối đa chỉ khoảng 50–70%**, kể cả với ngưỡng rộng. Lý do là đối chứng dương bị nhiễu: khoảng trống giữa hai chuyến đi có nhãn có thể chứa chuyến đi không được ghi nhãn, hoặc thiết bị không ghi. Con số này không phải độ nhạy thật của bộ phát hiện; nó chỉ dùng để so sánh giữa các ngưỡng.

**Quyết định:** giữ **200 m / 30 phút** (chế độ CENTROID).
- **30 phút** có bằng chứng trực tiếp: mốc lớn nhất giữ được các khoảng dừng ≥ 30 phút, với stay-point giả thấp.
- **200 m** là lựa chọn hợp lý: bao được độ phân tán lúc đứng yên, trùng với giá trị trong các nghiên cứu GeoLife, và không có bằng chứng nào cho thấy một mức khác tốt hơn.
- Việc gom các stay-point gần nhau thành **một địa điểm** (Home, Office) thuộc về classifier (giai đoạn 3), không thuộc về bán kính stay-point.

**Đuôi cửa sổ CENTROID (mục 4): giữ nguyên, không cắt.**

| | T0 (giữ nguyên) | T1 (cắt 1 lần) | T2 (cắt tới khi ổn định) |
|---|---|---|---|
| Stay-point có điểm xa tâm > 200 m | 68.5% | 56.1% | 13.7% |
| Tâm dịch chuyển (stay-point bị cắt), trung vị / P90 | — | 14 m / 33 m | 18 m / 45 m |
| Sai lệch giờ đến so với nhãn, trung vị tuyệt đối | 4.95 phút | 4.67 phút | 4.62 phút |
| Sai lệch giờ đi so với nhãn, trung vị tuyệt đối | 3.20 phút | 3.17 phút | 3.13 phút |
| Bắt được khoảng dừng ≥ 60 phút | 49.3% | 49.0% | 48.7% |
| Stay-point giả / 100 giờ đi xe | 3.47 | 3.28 | 3.24 |

- Cắt đuôi chỉ làm stay-point **"gọn" hơn về hình học**. Những gì classifier dùng gần như không đổi:
  - tâm dịch chuyển khoảng 14–18 m, nhỏ hơn nhiễu GPS (khoảng 34 m, notebook 01) và rất nhỏ so với bán kính 200 m;
  - giờ đến và giờ đi đổi chưa tới 0.3 phút;
  - độ nhạy giảm 0.3–0.6 điểm, stay-point giả giảm 5–7%.
- Phần đuôi rất ngắn về thời gian: trung vị 0.4 phút, P90 1.8 phút. Đó là vài chục mét lúc đang tới hoặc đang rời đi.
- Stay-point bắt đầu sớm hơn lúc chuyến đi trước kết thúc khoảng 4 phút, và kết thúc muộn hơn lúc chuyến sau bắt đầu khoảng 2 phút. Cắt đuôi **không sửa được** độ lệch này, vì các điểm đó nằm trong 200 m: đó là lúc đi bộ chậm quanh nơi đến, hoặc do giờ trên nhãn không chính xác.
- Thêm một quy tắc cắt đuôi nghĩa là thêm một ngưỡng và thêm độ phức tạp, mà không có lợi ích đo được. Vì vậy giữ nguyên cách tính của `src/`.